# Numerical implementation notes and scratch work

We aim to describe and implement drafts of the components for a (non-realtime) MPC algorithm for the motion cueing Stewart platform problem.

In [ ]:
from __future__ import annotations

In [ ]:
%matplotlib ipympl

import dataclasses
import functools
import typing as tp
import numpy as np
import sympy as sp
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import scipy.optimize as sci_opt
import IPython.display as ipd

import exp_mpc.stewart_min.const as const

## jax discrete integration

Simply, we want to compute recursively
\begin{align*}
  v_{k + 1} &= v_k + \Delta t \, a_{k + 1} \\
  x_{k + 1} &= x_k + \Delta t \, v_{k + 1}.
\end{align*}
Explicitly, this is because, definitionally, we have
$$
  a_{k + 1} = \frac{v_{k + 1} - v_k}{\Delta t} \quad\text{and}\quad v_{k + 1} = \frac{x_{k + 1} - x_k}{\Delta t}.
$$

In [ ]:
@jax.jit
def _discrete_1d_euler(
    x0: float,
    v0: float,
    a: jax.Array
) -> tuple[jax.Array, jax.Array]:
    v = jnp.cumsum(jnp.concatenate([jnp.array([v0]), const.dt * a]))
    x = jnp.cumsum(jnp.concatenate([jnp.array([x0]), const.dt * v.at[1:].get()]))
    return x, v


In [ ]:
a = jnp.arange(1, 10, dtype=float)
state0 = 0.0
v0 = 0.0
# %timeit _discrete_1d_euler(x0, v0, a)
x, v = _discrete_1d_euler(state0, v0, a)
x, v
# fig, ax = plt.subplots(2, 1, figsize=(8, 6))
# ax[0].plot(x, label="position")
# ax[1].plot(v, label="velocity")
# for a in ax:
#     a.legend()
#     a.grid()
#     a.set_xlabel("time")
# plt.show()

## jax discrete integration for non-uniform times

Eventually, we want to integrate over a non-uniform time grid.
Specifically, given the usual discrete time update rule, we want a single computation that projects the output into the future, e.g., we want a formula for $x_{k + n} f(x_k, v_k, a_k)$ where $a_k$ is applied on the index (time) intervals $[k, k + 1], \ldots, [k + n - 1, k + n]$.
It is not difficult to check that the following linear time-invariant linear ODE conforms with the update rule over a single time step $\Delta t$:
$$
\begin{bmatrix} \dot{x} \\ \dot{v} \end{bmatrix} = \begin{bmatrix} 0 & 1 \\ 0 & 0 \end{bmatrix} \begin{bmatrix} x \\ v \end{bmatrix} + \begin{bmatrix} \Delta t / 2 \\ 1 \end{bmatrix} a.
$$
where we have a single discrete update of
$$
x_{k + 1} := x_k + \Delta t \, v_{k + 1} = x_k + \Delta t \, v_k + (\Delta t)^2 \, a_k
$$
(no factor of $\frac{1}{2}$).

Note that the differential equation conforms with the discrete update after one time step.
To show that this holds after multiple time steps, we may proceed inductively, time-step by time-step, in the case of constant acceleration.
So,
$$
  v(t + T) = v(t) + T \, a \quad\text{and}\quad x(t + T) = x(t) + T \, v(t) + \frac{1}{2} \, T^2 \, a + \frac{1}{2} \, T \, \Delta t \, a
$$
e.g., for $T = k \, \Delta t$,
$$
  v(t + k \, \Delta t) = v(t) + k \, \Delta t \, a
$$
and
\begin{align*}
  x(t + k \, \Delta t) &= x(t) + k \, \Delta t \, v(t) + \frac{1}{2} \, k^2 \, (\Delta t)^2 \, a + \frac{1}{2} \, k \, (\Delta t)^2 \, a \\
  &= x(t) + k \, \Delta t \, v(t) + \frac{1}{2} \, (k^2 + k) \, (\Delta t)^2 \, a.
\end{align*}

For simple NumPy code, we observe that
$$
  x(t + k \, \Delta t) = x(t) + k \, \Delta t \, v(t + k \, \Delta t) + \frac{1}{2} \, (k - k^2) \, (\Delta t)^2 \, a.
$$
(Note the minus sign.)

In [ ]:
@jax.jit
def _discrete_1d_nonuniform_euler(
    x0: float,
    v0: float,
    a: jax.Array,
    gaps: jax.Array,
) -> tuple[jax.Array, jax.Array]:
    # cf. https://stackoverflow.com/questions/37726830/how-to-determine-if-a-number-is-any-type-of-int-core-or-numpy-signed-or-not
    assert jnp.issubdtype(gaps.dtype, jnp.integer)
    assert jnp.issubdtype(a.dtype, jnp.floating)
    a = jnp.ravel(a)
    gaps = jnp.ravel(gaps)
    assert a.shape == gaps.shape
    v = jnp.cumsum(jnp.concatenate([jnp.array([v0]), const.dt * gaps * a]))
    x_vel = const.dt * gaps * v.at[1:].get()
    x_acc = 0.5 * (gaps - gaps**2) * (const.dt**2) * a
    x = jnp.cumsum(jnp.concatenate([jnp.array([x0]), x_vel + x_acc]))
    return x, v

In [ ]:
gaps = jnp.array([1, 2, 3, 4, 5, 6, 7, 8, 9])
state0 = 0.0
v0 = 0.0
a = jnp.array([1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=float)
# %timeit _discrete_1d_nonuniform_euler(x0, v0, a, gaps)
_discrete_1d_nonuniform_euler(state0, v0, a, gaps)

In [ ]:
gaps = jnp.ones(9, dtype=int)
state0 = 0.0
v0 = 0.0
a = jnp.array([1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=float)
x_non, v_non = _discrete_1d_nonuniform_euler(state0, v0, a, gaps)
x_uni, v_uni = _discrete_1d_euler(state0, v0, a)
assert jnp.allclose(x_non, x_uni)
assert jnp.allclose(v_non, v_uni)


## brute force check for non-uniform discrete integration

In [ ]:
def _discrete_1d_nonuniform_brute(
    x0: float,
    v0: float,
    a: jax.Array,
    gaps: jax.Array,
) -> tuple[jax.Array, jax.Array]:
    assert jnp.issubdtype(gaps.dtype, jnp.integer)
    assert jnp.issubdtype(a.dtype, jnp.floating)
    a = jnp.ravel(a)
    gaps = jnp.ravel(gaps)
    assert a.shape == gaps.shape
    
    # all computations will be performed in the default python datatypes, which
    #  are quite expressive, but slow
    al = [float(acc) for acc in a]
    gl = [int(g) for g in gaps]
    assert len(al) == len(gl)
    al = sum([[al[i]] * gl[i] for i in range(len(al))], start=[])
    assert len(al) == sum(gl)
    x = [x0]
    v = [v0]
    for acc in al:
        v.append(v[-1] + const.dt * acc)
        x.append(x[-1] + const.dt * v[-1])
    
    indices = [0]
    for i in range(len(gl)):
        indices.append(indices[-1] + gl[i])
    x = [x[i] for i in indices]
    v = [v[i] for i in indices]
    return jnp.array(x), jnp.array(v)


In [ ]:
gaps = jnp.array([1, 2, 3, 4, 5, 6, 7, 8, 9])
state0 = 0.0
v0 = 0.0
a = jnp.array([1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=float)
x_brute, v_brute = _discrete_1d_nonuniform_brute(state0, v0, a, gaps)
x_non, v_non = _discrete_1d_nonuniform_euler(state0, v0, a, gaps)
assert jnp.allclose(v_brute, v_non)
assert jnp.allclose(x_brute, x_non)

## MPC bookkeeping

We want to bookkeep the variables productively for the Stewart platform MPC problem.
The SciPy optimization routines expect a flat array.
We produce two classes: a state class and a control class.
Because we have such a simple double integrator model, we implement something akin to a single shooting method.
This drastically reduces the number of variables in our optimization routine.

In [ ]:
@jax.tree_util.register_dataclass
@dataclasses.dataclass
class State:
    x: jax.Array
    y: jax.Array
    z: jax.Array
    roll: jax.Array
    pitch: jax.Array
    yaw: jax.Array
    x_dot: jax.Array
    y_dot: jax.Array
    z_dot: jax.Array
    roll_dot: jax.Array
    pitch_dot: jax.Array
    yaw_dot: jax.Array

In [ ]:
@jax.tree_util.register_dataclass
@dataclasses.dataclass
class Control:
    """Parallel array of control inputs."""

    x: jax.Array
    y: jax.Array
    z: jax.Array
    roll: jax.Array
    pitch: jax.Array
    yaw: jax.Array

    @classmethod
    def from_flat(cls, flat: jax.Array) -> "Control":
        """Convert a flat array to a control dataclass.

        We assume that the flat array is of the form:
            [x0, y0, z0, roll0, pitch0, yaw0,
             x1, y1, z1, roll1, pitch1, yaw1,
             ...]
        where the first three elements are the x, y and z coordinates of the
        first control point, the next three are the roll, pitch and yaw
        angles of the first control point, and so on for all control points.
        """
        assert flat.size % 6 == 0
        flat = jnp.reshape(flat, (-1, 6))
        return cls(
            x=flat[:, 0],
            y=flat[:, 1],
            z=flat[:, 2],
            roll=flat[:, 3],
            pitch=flat[:, 4],
            yaw=flat[:, 5],
        )

    def flatten(self) -> jax.Array:
        """Convert a control dataclass to a flat array."""
        return jnp.ravel(
            jnp.column_stack(
                [
                    self.x,
                    self.y,
                    self.z,
                    self.roll,
                    self.pitch,
                    self.yaw,
                ]
            )
        )

    def get_state(
        self,
        state0: jax.Array,
        gaps: tp.Optional[jax.Array] = None,
    ) -> State:
        """Convert a control dataclass to a state dataclass."""
        state0 = jnp.ravel(state0)
        assert state0.size == 12
        x0, y0, z0, roll0, pitch0, yaw0 = state0[:6]
        x_dot0, y_dot0, z_dot0, roll_dot0, pitch_dot0, yaw_dot0 = state0[6:]
        if gaps is None:
            gaps = jnp.ones(self.x.size, dtype=int)
        assert gaps.size == self.x.size
        _euler = _discrete_1d_nonuniform_euler
        x, x_dot = _euler(x0, x_dot0, self.x, gaps)
        y, y_dot = _euler(y0, y_dot0, self.y, gaps)
        z, z_dot = _euler(z0, z_dot0, self.z, gaps)
        roll, roll_dot = _euler(roll0, roll_dot0, self.roll, gaps)
        pitch, pitch_dot = _euler(pitch0, pitch_dot0, self.pitch, gaps)
        yaw, yaw_dot = _euler(yaw0, yaw_dot0, self.yaw, gaps)
        return State(
            x=x,
            y=y,
            z=z,
            roll=roll,
            pitch=pitch,
            yaw=yaw,
            x_dot=x_dot,
            y_dot=y_dot,
            z_dot=z_dot,
            roll_dot=roll_dot,
            pitch_dot=pitch_dot,
            yaw_dot=yaw_dot,
        )


## Cost Outline

We outline the components needed for our cost function.
1. Acceleration in the head (moving) frame
2. Angular velocity in the head (moving) frame
3. Boundary leg cost
4. Boundary joint angle cost
5. Yaw cost

These functions should be implemented in JAX, and they should include wrappers that input and output NumPy arrays (for the most compatibility with optimization libraries).
We need to be careful about scaling.

## Gradients and boundaries

We attempt to analyze the gradients of the cost function near the boundary to ensure that control variables push the robot away from their constraint violations. Note that we clip the reference acceleration and angular velocities, which gives us more provable control.